# Automatic entity type detection

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add the project root to the Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import AAAIM functions
from core import annotate_model, curate_model
from core.database_search import force_clear_chromadb, get_species_recommendations_rag
from utils.evaluation import (
    evaluate_single_model,
    evaluate_models_in_folder,
    print_evaluation_results,
    compare_results,
    process_saved_llm_responses
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# LLM configuration
llm_model = "openrouter/free"
# llm_model = "Llama-4-Maverick-17B-128E-Instruct-FP8"
# llm_model = "Llama-3.3-70B-Instruct"
# llm_model = "meta-llama/llama-3.3-70b-instruct:free"
# llm_model = "gpt-4.1-nano"

output_dir = "./autoType/"  # Output directory for results

In [2]:
test_model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000023.xml"
# test_model_file = "190_few_anno.xml"
# Check if test model exists
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

✓ Test model found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000023.xml


In [3]:
from core.model_info import format_prompt, find_species_with_annotations_and_qualifiers
# extract species that have chebi annotation
existing_annotations, qualifiers = find_species_with_annotations_and_qualifiers(test_model_file, 'chebi')
existing_annotations

{'Fru': ['15824'],
 'Glc': ['17634'],
 'HexP': ['16218', '15946', '14314', '18066'],
 'Suc6P': ['16308'],
 'Suc': ['17992'],
 'Sucvac': ['17992'],
 'glycolysis': ['28013'],
 'phos': ['18367'],
 'UDP': ['17659'],
 'ADP': ['16761'],
 'ATP': ['15422'],
 'Glcex': ['17634'],
 'Fruex': ['15824']}

In [6]:
# get the prompt for this model
prompt = format_prompt(test_model_file, existing_annotations.keys(), 'chemical')
print(prompt)

Now annotate these:
Chemical to annotate: Fru, Glc, HexP, Suc6P, Suc, Sucvac, glycolysis, phos, UDP, ADP, ATP, Glcex, Fruex
Model: "Rohwer2001_Sucrose"

// Reactions:
$Fruex -> Fru
$Glcex -> Glc
$ATP + Glc -> HexP + $ADP
Fru + $ATP -> HexP + $ADP
Fru + $ATP -> HexP + $ADP
2 HexP -> $UDP + Suc6P
Suc6P -> Suc + $phos
HexP + Fru -> Suc + $UDP
Suc -> Fru + Glc
HexP -> $glycolysis
Suc -> $Sucvac

// Notes:
"SBML
Level 2 code generated for the JWS Online project by Jacky Snoep using
PySCeS
.
Run this model online at
http://jjj.biochem.sun.ac.za
.
Web-based modelling using JWS Online
, Bioinformatics, 20:2143-2144.
For more information see the
.
for more information."

Return up to 3 standardized names or common synonyms for each chemical, ranked by likelihood. Provide components names for complexes, which may exceed the limit of 3.
Use the below format, do not include any other text except the synonyms, and give short reasons for all chemicals after 'Reason:' by the end.

SpeciesA: "name1", 

In [5]:
result_df = evaluate_single_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method = 'direct',
    top_k = 5,
    entity_type='chemical',
    database='chebi',
    save_llm_results=False,
    output_dir=output_dir,
    verbose=True
)

2025-11-24 13:13:33,786 - INFO - Evaluating model: BIOMD0000000023.xml
2025-11-24 13:13:33,889 - INFO - Evaluating 13 entities in BIOMD0000000023.xml
2025-11-24 13:13:37,702 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"


LLM response: 
Fru: "fructose", "D-fructose", "beta-D-fructose"
Glc: "glucose", "D-glucose", "alpha-D-glucose"
HexP: "hexose phosphate", "fructose-6-phosphate", "glucose-6-phosphate"
Suc6P: "sucrose-6-phosphate", "sucrose phosphate", "6-phosphosucrose"
Suc: "sucrose", "alpha-D-glucopyranosyl beta-D-fructofuranoside", "beta-D-fructofuranosyl alpha-D-glucopyranoside"
Sucvac: "sucrose", "vacuolar sucrose", "sucrose in vacuole"
Glcex: "glucose", "extracellular glucose", "external glucose"
Fruex: "fructose", "extracellular fructose", "external fructose"
phos: "phosphate", "inorganic phosphate", "orthophosphate"
UDP: "uridine diphosphate", "uridine 5'-diphosphate", "UDP"
ADP: "adenosine diphosphate", "adenosine 5'-diphosphate", "ADP"
ATP: "adenosine triphosphate", "adenosine 5'-triphosphate", "ATP"
glycolysis: "glycolytic pathway", "glycolysis intermediate", "UNK"
Reason: The model appears to be a metabolic model of sucrose metabolism, with reactions involving glucose, fructose, and sucrose.

In [6]:
result_df

,model,species_id,display_name,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,match_score,...,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier,detected_entity_type
0,BIOMD0000000023.xml,Fru,,"[fructose, D-fructose, beta-D-fructose]",Chunk 1: The model appears to be a metabolic m...,[15824],D-fructose,"[28645, 28757, 15824, 37721, 48095]","beta-D-fructofuranose, fructose, D-fructose, D...","[0.6666666666666666, 0.3333333333333333, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
1,BIOMD0000000023.xml,Glc,,"[glucose, D-glucose, alpha-D-glucose]",,[17634],D-glucose,"[4167, 42758, 17234, 17634, 17925]","D-glucopyranose, aldehydo-D-glucose, glucose, ...","[0.6666666666666666, 0.6666666666666666, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
2,BIOMD0000000023.xml,HexP,,"[hexose phosphate, fructose-6-phosphate, gluco...",,"[16218, 15946, 14314, 18066]","beta-D-glucose 1-phosphate, keto-D-fructose 6-...","[47878, 16084, 15946, 88003]","hexose phosphate, beta-D-fructofuranose 6-phos...","[0.3333333333333333, 0.3333333333333333, 0.333...",...,0.25,0.25,1,5.373753,3.784719,1.589034,None,None,"hasVersion, hasVersion, hasVersion, hasVersion",unknown
3,BIOMD0000000023.xml,Suc6P,,"[sucrose-6-phosphate, sucrose phosphate, 6-pho...",,[16308],sucrose 6(F)-phosphate,"[131603, 16308]","sucrose 6(G)-phosphate, sucrose 6(F)-phosphate","[0.6666666666666666, 0.6666666666666666]",...,1.00,0.50,1,5.373753,3.784719,1.589034,None,None,is,unknown
4,BIOMD0000000023.xml,Suc,,"[sucrose, alpha-D-glucopyranosyl beta-D-fructo...",,[17992],sucrose,[17992],sucrose,[0.6666666666666666],...,1.00,1.00,1,5.373753,3.784719,1.589034,None,None,is,unknown
5,BIOMD0000000023.xml,Sucvac,,"[sucrose, vacuolar sucrose, sucrose in vacuole]",,[17992],sucrose,[17992],sucrose,[0.3333333333333333],...,1.00,1.00,1,5.373753,3.784719,1.589034,None,None,is,unknown
6,BIOMD0000000023.xml,glycolysis,,"[glycolytic pathway, glycolysis intermediate, ...",,[28013],"beta-D-fructofuranose 1,6-bisphosphate",[],NA,[],...,0.00,0.00,0,5.373753,3.784719,1.589034,None,None,is,unknown
7,BIOMD0000000023.xml,phos,,"[phosphate, inorganic phosphate, orthophosphate]",,[18367],phosphate(3-),"[18367, 26020, 26078, 35780, 43474]","phosphate(3-), phosphate, phosphoric acid, pho...","[0.6666666666666666, 0.3333333333333333, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
8,BIOMD0000000023.xml,UDP,,"[uridine diphosphate, uridine 5'-diphosphate, ...",,[17659],UDP,"[17659, 58223]","UDP, UDP(3-)","[1.0, 0.6666666666666666]",...,1.00,0.50,1,5.373753,3.784719,1.589034,None,None,is,unknown
9,BIOMD0000000023.xml,ADP,,"[adenosine diphosphate, adenosine 5'-diphospha...",,[16761],ADP,"[16761, 456216, 177051, 73342]","ADP, ADP(3-), Adenosine_Diphosphate, Ala-Asp-Pro","[0.6666666666666666, 0.6666666666666666, 0.333...",...,1.00,0.25,1,5.373753,3.784719,1.589034,None,None,is,unknown


# Test on auto detection

In [7]:
import sys
import importlib
importlib.reload(sys.modules['core.model_info'])
importlib.reload(sys.modules['utils.evaluation'])
from core.model_info import format_prompt, find_species_with_annotations_and_qualifiers
from utils.evaluation import evaluate_single_model

In [8]:
prompt = format_prompt(test_model_file, existing_annotations.keys(), 'auto')
print(prompt)

Now annotate these species:
Species to annotate: Fru, Glc, HexP, Suc6P, Suc, Sucvac, glycolysis, phos, UDP, ADP, ATP, Glcex, Fruex
Model: "Rohwer2001_Sucrose"

// Reactions:
$Fruex -> Fru
$Glcex -> Glc
$ATP + Glc -> HexP + $ADP
Fru + $ATP -> HexP + $ADP
Fru + $ATP -> HexP + $ADP
2 HexP -> $UDP + Suc6P
Suc6P -> Suc + $phos
HexP + Fru -> Suc + $UDP
Suc -> Fru + Glc
HexP -> $glycolysis
Suc -> $Sucvac

// Notes:
"SBML
Level 2 code generated for the JWS Online project by Jacky Snoep using
PySCeS
.
Run this model online at
http://jjj.biochem.sun.ac.za
.
Web-based modelling using JWS Online
, Bioinformatics, 20:2143-2144.
For more information see the
.
for more information."

For each species, determine its entity type (auto, chemical, gene, protein, complex, unknown).
Return up to 3 standardized names or common synonyms for each species, ranked by likelihood. Provide components names for complexes, which may exceed the limit of 3.
Specify the entity type in parentheses after each species ID.

In [9]:
result_df = evaluate_single_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method = 'rag',
    top_k = 3,
    entity_type='auto',
    database=["chebi", "uniprot"],
    save_llm_results=False,
    output_dir=output_dir,
    verbose=True
)

2026-08-28 16:29:38,955 - INFO - Evaluating model: BIOMD0000000023.xml
2026-08-28 16:29:38,957 - INFO - Using default tax_id: 9606
2026-08-28 16:29:39,051 - INFO - Evaluating 13 entities in BIOMD0000000023.xml
2026-08-28 16:30:44,500 - INFO - Detected entity types: {'chemical': 11, 'unknown': 2}
2026-08-28 16:30:44,503 - INFO - Searching chebi for 11 chemical entities
2026-08-28 16:30:44,535 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-08-28 16:30:44,928 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-28 16:30:48,923 - WARNING - There are 2 species with unknown entity type: ['Sucvac', 'glycolysis']


In [10]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000023.xml,Fru,,chemical,"[fructose, D-fructose, fructopyranose]","Chunk 1: Fru, Glc, HexP, Suc6P, Suc, phos, UDP...",[15824],D-fructose,"[28757, 28645, 48095]","fructose, beta-D-fructofuranose, keto-D-fructose",...,1.0,0.00,0.000000,1,69.829449,65.406248,4.423201,9606,None,is
1,BIOMD0000000023.xml,Glc,,chemical,"[glucose, D-glucose, glucopyranose]",,[17634],D-glucose,"[4167, 42758, 17234]","D-glucopyranose, aldehydo-D-glucose, glucose",...,1.0,0.00,0.000000,1,69.829449,65.406248,4.423201,9606,None,is
2,BIOMD0000000023.xml,HexP,,chemical,"[hexose phosphate, glucose-6-phosphate, fructo...",,"[16218, 15946, 14314, 18066]","beta-D-glucose 1-phosphate, keto-D-fructose 6-...","[47878, 16084, 15946]","hexose phosphate, beta-D-fructofuranose 6-phos...",...,1.0,0.25,0.333333,1,69.829449,65.406248,4.423201,9606,None,"hasVersion, hasVersion, hasVersion, hasVersion"
3,BIOMD0000000023.xml,Suc6P,,chemical,"[sucrose-6-phosphate, sucrose 6-phosphate, C12...",,[16308],sucrose 6(F)-phosphate,"[131603, 16308, 84067]","sucrose 6(G)-phosphate, sucrose 6(F)-phosphate...",...,0.5,1.00,0.333333,1,69.829449,65.406248,4.423201,9606,None,is
4,BIOMD0000000023.xml,Suc,,chemical,"[sucrose, table sugar, β-D-glucopyranosyl-(1→2...",,[17992],sucrose,"[17992, 32528, 50661]","sucrose, turanose, loliose",...,0.5,1.00,0.333333,1,69.829449,65.406248,4.423201,9606,None,is
5,BIOMD0000000023.xml,phos,,chemical,"[phosphate, inorganic phosphate, Pi]",,[18367],phosphate(3-),"[26078, 35780, 43474]","phosphoric acid, phosphate ion, hydrogenphosphate",...,1.0,0.00,0.000000,1,69.829449,65.406248,4.423201,9606,None,is
6,BIOMD0000000023.xml,UDP,,chemical,"[uridine diphosphate, UDP, uridine-5′-diphosph...",,[17659],UDP,"[17659, 15713, 58223]","UDP, UTP, UDP(3-)",...,0.5,1.00,0.333333,1,69.829449,65.406248,4.423201,9606,None,is
7,BIOMD0000000023.xml,ADP,,chemical,"[adenosine diphosphate, ADP, adenosine-5′-diph...",,[16761],ADP,"[16761, 456216, 73342]","ADP, ADP(3-), Ala-Asp-Pro",...,0.5,1.00,0.333333,1,69.829449,65.406248,4.423201,9606,None,is
8,BIOMD0000000023.xml,ATP,,chemical,"[adenosine triphosphate, ATP, adenosine-5′-tri...",,[15422],ATP,"[15422, 30616, 27775]","ATP, ATP(4-), P(1),P(3)-bis(5'-adenosyl) triph...",...,0.5,1.00,0.333333,1,69.829449,65.406248,4.423201,9606,None,is
9,BIOMD0000000023.xml,Glcex,,chemical,"[extracellular glucose, Glc_ex, glucose (extra...",,[17634],D-glucose,"[4167, 42758, 17234]","D-glucopyranose, aldehydo-D-glucose, glucose",...,1.0,0.00,0.000000,1,69.829449,65.406248,4.423201,9606,None,is


In [8]:
from core.llm_interface import SYSTEM_PROMPT, query_llm, parse_llm_response, get_system_prompt
llm_response = """
Ca_cyt (chemical): "calcium(2+)", "calcium ion", "Ca2+"
CaER (chemical): "calcium", "endoplasmic reticulum calcium", "Ca2+ in endoplasmic reticulum"
CaM (chemical): "calmodulin", "calcium-modulated protein", "CaM protein"
CaPr (complex): "calcium-protein complex", "protein-calcium complex", "calcium bound protein"

Reason: The model "Marhl2000_CaOscillations" suggests a focus on calcium oscillations
"""
chunk_synonyms_dict, chunk_entity_type_dict, chunk_reason = parse_llm_response(llm_response)

In [29]:
# Test with a single model using automatic entity type detection
recommendations_df, metrics = annotate_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method="direct",
    top_k=3,
    entity_type="auto",
    database=["chebi", "uniprot"]
)

2025-11-17 14:57:08,492 - INFO - Starting annotation for model: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000039.xml
2025-11-17 14:57:08,493 - INFO - Using LLM model: Llama-3.3-70B-Instruct
2025-11-17 14:57:08,493 - INFO - Using method: direct for database search
2025-11-17 14:57:08,494 - INFO - Entity type: auto, Database: ['chebi', 'uniprot']
2025-11-17 14:57:08,494 - INFO - >>>Step 1: Getting species from model...<<<
2025-11-17 14:57:08,500 - INFO - Found 5 species in model
2025-11-17 14:57:08,501 - WARNING - Entity type auto with database ['chebi', 'uniprot'] not yet supported
2025-11-17 14:57:08,501 - INFO - Annotate all 5 entities
2025-11-17 14:57:08,501 - INFO - >>>Step 2: Extracting model context...<<<
2025-11-17 14:57:08,526 - INFO - Extracted context for model: Marhl2000_CaOscillations
2025-11-17 14:57:08,526 - INFO - >>>Step 3: Querying LLM (Llama-3.3-70B-Instruct)...<<<
2025-11-17 14:57:08,546 - WARNING - Auto entity type detection not yet implemented
202

## uniprot

In [2]:
test_model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106/BIOMD0000000137.xml"
tax_id = 9606
# Check if test model exist
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

✓ Test model found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106/BIOMD0000000137.xml


In [3]:
# Running with auto entity type
result_df = evaluate_single_model(
    model_file=test_model_file,
    tax_id=tax_id,
    llm_model=llm_model,
    entity_type='auto',
    database='uniprot',
    save_llm_results=False,
    output_dir=output_dir
)

2025-12-11 17:30:12,649 - INFO - Evaluating model: BIOMD0000000137.xml
2025-12-11 17:30:12,650 - INFO - Using organism-specific search for tax_id: 9606
2025-12-11 17:30:12,676 - INFO - Evaluating 18 entities in BIOMD0000000137.xml
2025-12-11 17:30:17,355 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-11 17:30:17,366 - INFO - Detected entity types: {'chemical': 1, 'protein': 13, 'complex': 4}
2025-12-11 17:30:17,366 - WARNING - No valid database found for entity type 'chemical' in allowed databases: ['uniprot']
2025-12-11 17:30:17,366 - WARNING - No valid database found for entity type 'chemical' in ['uniprot'] for 1 species
2025-12-11 17:30:17,367 - INFO - Searching uniprot for 13 protein entities
2025-12-11 17:30:17,643 - INFO - Searching all databases ['uniprot'] for 4 complex entities


In [4]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000137.xml,x1,Insulin,chemical,"[Insulin, human insulin, insulin hormone]","Chunk 1: x1 is a hormone (chemical), x2 is a r...",[P01308],INS,[],NA,...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,is
1,BIOMD0000000137.xml,x2,Unbound Insulin Receptor,protein,"[Insulin receptor, INSR, CD220]",,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,4.963253,4.643137,0.320116,9606,None,isVersionOf
2,BIOMD0000000137.xml,x6,Unbound unphosphorylated intracellular receptor,protein,"[Insulin receptor substrate, IRS, IRS-1]",,[P06213],INSR,"[P01308, P41252, P35568]","INS, IARS1, IRS1",...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf
3,BIOMD0000000137.xml,x7,Phosphorylated twice bound intracellular receptor,protein,"[Phosphorylated IRS, pIRS, IRS-1]",,[P06213],INSR,"[P41252, P35568]","IARS1, IRS1",...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf
4,BIOMD0000000137.xml,x8,Phosphorylated once bound intracellular receptor,protein,"[Phosphorylated IRS, pIRS, IRS-1]",,[P06213],INSR,"[P41252, P35568]","IARS1, IRS1",...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf
5,BIOMD0000000137.xml,x9,Unphosphorylated IRS1,protein,"[IRS1, Insulin receptor substrate 1, IRS-1]",,[P35568],IRS1,[P35568],IRS1,...,1.0,1.0,1.0,1,4.963253,4.643137,0.320116,9606,None,isVersionOf
6,BIOMD0000000137.xml,x10,Phosphorylated IRS1,protein,"[Phosphorylated IRS1, pIRS1, IRS-1]",,[P35568],IRS1,[P35568],IRS1,...,1.0,1.0,1.0,1,4.963253,4.643137,0.320116,9606,None,isVersionOf
7,BIOMD0000000137.xml,x11,PI3 Kinase,protein,"[PI3K, Phosphatidylinositol 3-kinase, PI3 kinase]",,[Q8WYR1],PIK3R5,[P19957],PI3,...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf
8,BIOMD0000000137.xml,x16,Unactivated Akt,protein,"[Akt, Protein kinase B, PKB]",,[Q9Y243],AKT3,[P31749],AKT1,...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf
9,BIOMD0000000137.xml,x17,Activated Akt,protein,"[Phosphorylated Akt, pAkt, Akt1]",,[Q9Y243],AKT3,[P31749],AKT1,...,0.0,0.0,0.0,0,4.963253,4.643137,0.320116,9606,None,isVersionOf


In [5]:
# Running with auto entity type
result_df = evaluate_single_model(
    model_file=test_model_file,
    tax_id=tax_id,
    llm_model=llm_model,
    entity_type='auto',
    database=['chebi', 'uniprot'],
    save_llm_results=False,
    output_dir=output_dir
)

2025-12-10 12:49:39,955 - INFO - Evaluating model: BIOMD0000000137.xml
2025-12-10 12:49:39,955 - INFO - Using organism-specific search for tax_id: 9606
2025-12-10 12:49:40,043 - INFO - Evaluating 21 entities in BIOMD0000000137.xml
2025-12-10 12:49:45,437 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-10 12:49:45,438 - INFO - Detected entity types: {'chemical': 4, 'protein': 16, 'complex': 1}
2025-12-10 12:49:45,439 - INFO - Searching chebi for 4 chemical entities
2025-12-10 12:49:46,359 - INFO - Searching uniprot for 16 protein entities
2025-12-10 12:49:46,501 - INFO - Searching all databases ['chebi', 'uniprot'] for 1 complex entities


In [6]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000137.xml,x13,"PI3,4,5P3",chemical,"[Phosphatidylinositol (3,4,5)-trisphosphate, P...",Chunk 1: The entity types were determined base...,[16618],"1-phosphatidyl-1D-myo-inositol 3,4,5-trisphosp...",[16618],"1-phosphatidyl-1D-myo-inositol 3,4,5-trisphosp...",...,1.0,1.0,1.0,1,6.49642,5.355494,1.140926,9606,None,is
1,BIOMD0000000137.xml,x14,"PI4,5P2",chemical,"[Phosphatidylinositol 4,5-bisphosphate, PIP2, ...",,[18348],"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate","[18348, 232300]","1-phosphatidyl-1D-myo-inositol 4,5-bisphosphat...",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,is
2,BIOMD0000000137.xml,x15,"PI3,4P2",chemical,"[Phosphatidylinositol (3,4)-bisphosphate, PI(3...",,[16152],"1-phosphatidyl-1D-myo-inositol 3,4-bisphosphate",[],NA,...,0.0,0.0,0.0,0,6.49642,5.355494,1.140926,9606,None,is
3,BIOMD0000000137.xml,x1,Insulin,chemical,"[Insulin, human insulin, insulin hormone]",,[P01308],INS,"[5931, 145810]","insulin (human), insulin",...,0.0,0.0,0.0,0,6.49642,5.355494,1.140926,9606,None,is
4,BIOMD0000000137.xml,x2,Unbound Insulin Receptor,protein,"[Insulin receptor, INSR, IR]",,[P06213],INSR,"[P06213, P01308]","INSR, INS",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf
5,BIOMD0000000137.xml,x3,Unphosphorylated once bound receptor,protein,"[Unphosphorylated insulin receptor, Insulin re...",,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf
6,BIOMD0000000137.xml,x5,Phosphorylated once bound receptor,protein,"[Phosphorylated insulin receptor, Insulin rece...",,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf
7,BIOMD0000000137.xml,x4,Phosphorylated twice bound receptor,protein,"[Phosphorylated insulin receptor, Insulin rece...",,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf
8,BIOMD0000000137.xml,x6,Unbound unphosphorylated intracellular receptor,protein,[Unbound unphosphorylated intracellular recept...,,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf
9,BIOMD0000000137.xml,x7,Phosphorylated twice bound intracellular receptor,protein,[Phosphorylated twice bound intracellular rece...,,[P06213],INSR,"[P01308, P06213]","INS, INSR",...,0.5,1.0,0.5,1,6.49642,5.355494,1.140926,9606,None,isVersionOf


# Batch evaluation
## chebi

In [2]:
model_dir = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106"
# model_dir = "test_models"
# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} XML files")
    # print(f"  - Will test first {min(num_models_to_test, len(model_files))} models")

✓ Model directory found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106
  - Found 1075 XML files


In [ ]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv",
    start_at=1
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 10,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts_updated.csv",
    start_at=1
)

In [ ]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 10,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-3_top10_autoType.csv",
    start_at=1
)

In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-3_top10_autoType.csv")

Filtered results to 9312 entries that exist in reference: /Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv
Number of models assessed: 303
Number of models with predictions: 303
Number of annotations evaluated: 9312
Average accuracy (per model): 0.94
Ave. recall (formula): 0.94
Ave. precision (formula): 0.23
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.94
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.74
Ave. precision (exact, per species): 0.11
Ave. total time (per model): 21.80
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 21.00
Ave. LLM time (per element, per model): 0.68
Average number of predictions per species: 9.94


## Uniprot

In [4]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='uniprot',
    method="direct",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_uniprot_direct_llama-4_top3_autoType_complex_improved.csv",
    start_at=1,
    tax_id = 9606
)

2025-12-07 13:30:55,663 - WARNING - Skipping BIOMD0000000001.xml - no results generated
2025-12-07 13:30:55,674 - WARNING - Skipping BIOMD0000000002.xml - no results generated
2025-12-07 13:30:55,680 - WARNING - Skipping BIOMD0000000003.xml - no results generated
2025-12-07 13:30:55,684 - WARNING - Skipping BIOMD0000000004.xml - no results generated
2025-12-07 13:30:55,691 - WARNING - Skipping BIOMD0000000005.xml - no results generated
2025-12-07 13:30:55,695 - WARNING - Skipping BIOMD0000000006.xml - no results generated
2025-12-07 13:30:55,707 - WARNING - Skipping BIOMD0000000007.xml - no results generated
2025-12-07 13:30:55,713 - WARNING - Skipping BIOMD0000000008.xml - no results generated
2025-12-07 13:30:55,724 - WARNING - Skipping BIOMD0000000009.xml - no results generated
2025-12-07 13:30:55,730 - WARNING - Skipping BIOMD0000000010.xml - no results generated
2025-12-07 13:30:55,744 - WARNING - Skipping BIOMD0000000011.xml - no results generated
2025-12-07 13:30:55,751 - WARNIN

LLM results will be saved to: ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251207_1330
Saved configuration to ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251207_1330/config.txt
Evaluating 1/1075: BIOMD0000000001.xml
Evaluating 2/1075: BIOMD0000000002.xml
Evaluating 3/1075: BIOMD0000000003.xml
Evaluating 4/1075: BIOMD0000000004.xml
Evaluating 5/1075: BIOMD0000000005.xml
Evaluating 6/1075: BIOMD0000000006.xml
Evaluating 7/1075: BIOMD0000000007.xml
Evaluating 8/1075: BIOMD0000000008.xml
Evaluating 9/1075: BIOMD0000000009.xml
Evaluating 10/1075: BIOMD0000000010.xml
Evaluating 11/1075: BIOMD0000000011.xml
Evaluating 12/1075: BIOMD0000000012.xml
Evaluating 13/1075: BIOMD0000000013.xml
Evaluating 14/1075: BIOMD0000000014.xml
Evaluating 15/1075: BIOMD0000000015.xml


2025-12-07 13:30:55,859 - WARNING - Skipping BIOMD0000000015.xml - no results generated
2025-12-07 13:30:55,865 - WARNING - Skipping BIOMD0000000016.xml - no results generated
2025-12-07 13:30:55,875 - WARNING - Skipping BIOMD0000000017.xml - no results generated


Evaluating 16/1075: BIOMD0000000016.xml
Evaluating 17/1075: BIOMD0000000017.xml
Evaluating 18/1075: BIOMD0000000018.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251207_1330/BIOMD0000000018.txt
Evaluating 19/1075: BIOMD0000000019.xml


2025-12-07 13:31:03,599 - WARNING - Skipping BIOMD0000000020.xml - no results generated
2025-12-07 13:31:03,608 - WARNING - Skipping BIOMD0000000021.xml - no results generated
2025-12-07 13:31:03,619 - WARNING - Skipping BIOMD0000000022.xml - no results generated
2025-12-07 13:31:03,628 - WARNING - Skipping BIOMD0000000023.xml - no results generated
2025-12-07 13:31:03,631 - WARNING - Skipping BIOMD0000000024.xml - no results generated
2025-12-07 13:31:03,635 - WARNING - Skipping BIOMD0000000025.xml - no results generated
2025-12-07 13:31:03,641 - WARNING - Skipping BIOMD0000000026.xml - no results generated
2025-12-07 13:31:03,645 - WARNING - Skipping BIOMD0000000027.xml - no results generated
2025-12-07 13:31:03,655 - WARNING - Skipping BIOMD0000000028.xml - no results generated
2025-12-07 13:31:03,660 - WARNING - Skipping BIOMD0000000029.xml - no results generated
2025-12-07 13:31:03,670 - WARNING - Skipping BIOMD0000000030.xml - no results generated
2025-12-07 13:31:03,674 - WARNIN

...

Evaluating 1068/1075: BIOMD0000001094.xml
Evaluating 1069/1075: BIOMD0000001095.xml


2025-12-07 14:11:43,350 - WARNING - Skipping BIOMD0000001095.xml - no results generated
2025-12-07 14:11:43,695 - WARNING - Skipping BIOMD0000001096.xml - no results generated
2025-12-07 14:11:43,903 - WARNING - Skipping BIOMD0000001097.xml - no results generated


Evaluating 1070/1075: BIOMD0000001096.xml
Evaluating 1071/1075: BIOMD0000001097.xml


2025-12-07 14:11:44,044 - WARNING - Skipping BIOMD0000001098.xml - no results generated
2025-12-07 14:11:44,215 - WARNING - Skipping BIOMD0000001099.xml - no results generated
2025-12-07 14:11:44,223 - WARNING - Skipping BIOMD0000001102.xml - no results generated
2025-12-07 14:11:44,231 - WARNING - Skipping BIOMD0000001103.xml - no results generated


Evaluating 1072/1075: BIOMD0000001098.xml
Evaluating 1073/1075: BIOMD0000001099.xml
Evaluating 1074/1075: BIOMD0000001102.xml
Evaluating 1075/1075: BIOMD0000001103.xml


In [6]:
print_evaluation_results("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType_complex_improved.csv", ref_results_csv=None, entity_types=['complex'])

Showing all results
Filtering results by entity types: ['complex']
Number of models assessed: 204
Number of models with predictions: 174
Number of annotations evaluated: 2117
Average accuracy (per model): 0.61
Ave. recall (formula): 0.46
Ave. precision (formula): 0.50
Ave. recall (exact): 0.46
Ave. precision (exact): 0.50
Average accuracy (per species): 0.57
Ave. recall (formula, per species): 0.40
Ave. precision (formula, per species): 0.45
Ave. recall (exact, per species): 0.40
Ave. precision (exact, per species): 0.45
Ave. total time (per model): 10.20
Ave. total time (per element, per model): 0.98
Ave. LLM time (per model): 10.00
Ave. LLM time (per element, per model): 0.96
Average number of predictions per species: 1.38


In [7]:
print_evaluation_results("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType_complex_improved.csv", ref_results_csv=None, entity_types=['protein'])

Showing all results
Filtering results by entity types: ['protein']
Number of models assessed: 277
Number of models with predictions: 259
Number of annotations evaluated: 2628
Average accuracy (per model): 0.62
Ave. recall (formula): 0.60
Ave. precision (formula): 0.58
Ave. recall (exact): 0.60
Ave. precision (exact): 0.58
Average accuracy (per species): 0.59
Ave. recall (formula, per species): 0.57
Ave. precision (formula, per species): 0.55
Ave. recall (exact, per species): 0.57
Ave. precision (exact, per species): 0.55
Ave. total time (per model): 8.27
Ave. total time (per element, per model): 0.87
Ave. LLM time (per model): 8.11
Ave. LLM time (per element, per model): 0.85
Average number of predictions per species: 0.81


In [12]:
print_evaluation_results("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType_complex_improved.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 272
Number of models with predictions: 235
Number of annotations evaluated: 2810
Average accuracy (per model): 0.52
Ave. recall (formula): 0.51
Ave. precision (formula): 0.48
Ave. recall (exact): 0.51
Ave. precision (exact): 0.48
Average accuracy (per species): 0.51
Ave. recall (formula, per species): 0.51
Ave. precision (formula, per species): 0.48
Ave. recall (exact, per species): 0.51
Ave. precision (exact, per species): 0.48
Ave. total time (per model): 8.15
Ave. total time (per element, per model): 0.79
Ave. LLM time (per model): 7.99
Ave. LLM time (per element, per model): 0.77
Average number of predictions per species: 0.72


In [13]:
print_evaluation_results("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 472
Number of models with predictions: 352
Number of annotations evaluated: 7825
Average accuracy (per model): 0.07
Ave. recall (formula): 0.00
Ave. precision (formula): 0.00
Ave. recall (exact): 0.07
Ave. precision (exact): 0.06
Average accuracy (per species): 0.03
Ave. recall (formula, per species): 0.00
Ave. precision (formula, per species): 0.00
Ave. recall (exact, per species): 0.03
Ave. precision (exact, per species): 0.02
Ave. total time (per model): 11.71
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 11.51
Ave. LLM time (per element, per model): 0.69
Average number of predictions per species: 0.32


In [19]:
model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioDivine/'

# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.sbml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} SBML files")

✓ Model directory found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioDivine/
  - Found 190 SBML files


In [ ]:
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    top_k=3,
    entity_type='auto',
    database='uniprot',
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biodivine_uniprot_direct_llama-3_top5_9606_autoType.csv",
    start_at=1,
    tax_id = 9606
)

In [22]:
print_evaluation_results("autoType/biodivine_uniprot_direct_llama-3_top5_9606_autoType.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 15
Number of models with predictions: 8
Number of annotations evaluated: 605
Average accuracy (per model): 0.00
Ave. recall (formula): 0.00
Ave. precision (formula): 0.00
Ave. recall (exact): 0.00
Ave. precision (exact): 0.00
Average accuracy (per species): 0.00
Ave. recall (formula, per species): 0.00
Ave. precision (formula, per species): 0.00
Ave. recall (exact, per species): 0.00
Ave. precision (exact, per species): 0.00
Ave. total time (per model): 14.54
Ave. total time (per element, per model): 0.36
Ave. LLM time (per model): 14.34
Ave. LLM time (per element, per model): 0.36
Average number of predictions per species: 0.08


## chebi + uniprot

In [ ]:
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database=['chebi', 'uniprot'],
    method="direct",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv",
    start_at=1,
    tax_id = 9606
)

In [2]:
from utils.evaluation import recalculate_statistics
df = recalculate_statistics("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv", "autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType_recal.csv")

Reading results from: autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv
Recalculated statistics saved to: autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType_recal.csv
Processed 17510 rows

Summary statistics (recalculated):
  Average accuracy: 0.685
  Average recall (formula): 0.673
  Average precision (formula): 0.628
  Average recall (exact): 0.460
  Average precision (exact): 0.450


In [13]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv", ref_results_csv=None, entity_types=['chemical'])

Showing all results
Filtering results by entity types: ['chemical']
Number of models assessed: 404
Number of models with predictions: 341
Number of annotations evaluated: 11613
Average accuracy (per model): 0.64
Ave. recall (formula): 0.64
Ave. precision (formula): 0.58
Ave. recall (exact): 0.55
Ave. precision (exact): 0.30
Average accuracy (per species): 0.86
Ave. recall (formula, per species): 0.86
Ave. precision (formula, per species): 0.80
Ave. recall (exact, per species): 0.53
Ave. precision (exact, per species): 0.53
Ave. total time (per model): 18.65
Ave. total time (per element, per model): 0.65
Ave. LLM time (per model): 17.26
Ave. LLM time (per element, per model): 0.60
Average number of predictions per species: 1.82


In [14]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv", ref_results_csv=None, entity_types=['protein'])

Showing all results
Filtering results by entity types: ['protein']
Number of models assessed: 313
Number of models with predictions: 282
Number of annotations evaluated: 2714
Average accuracy (per model): 0.57
Ave. recall (formula): 0.00
Ave. precision (formula): 0.00
Ave. recall (exact): 0.55
Ave. precision (exact): 0.53
Average accuracy (per species): 0.59
Ave. recall (formula, per species): 0.00
Ave. precision (formula, per species): 0.00
Ave. recall (exact, per species): 0.57
Ave. precision (exact, per species): 0.54
Ave. total time (per model): 22.96
Ave. total time (per element, per model): 2.65
Ave. LLM time (per model): 21.37
Ave. LLM time (per element, per model): 2.46
Average number of predictions per species: 0.86


In [15]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv", ref_results_csv=None, entity_types=['complex'])

Showing all results
Filtering results by entity types: ['complex']
Number of models assessed: 265
Number of models with predictions: 153
Number of annotations evaluated: 2855
Average accuracy (per model): 0.12
Ave. recall (formula): 0.13
Ave. precision (formula): 0.12
Ave. recall (exact): 0.10
Ave. precision (exact): 0.07
Average accuracy (per species): 0.14
Ave. recall (formula, per species): 0.14
Ave. precision (formula, per species): 0.15
Ave. recall (exact, per species): 0.10
Ave. precision (exact, per species): 0.09
Ave. total time (per model): 26.40
Ave. total time (per element, per model): 2.45
Ave. LLM time (per model): 24.54
Ave. LLM time (per element, per model): 2.28
Average number of predictions per species: 0.89


In [4]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 552
Number of models with predictions: 521
Number of annotations evaluated: 17510
Average accuracy (per model): 0.57
Ave. recall (formula): 0.38
Ave. precision (formula): 0.35
Ave. recall (exact): 0.51
Ave. precision (exact): 0.36
Average accuracy (per species): 0.69
Ave. recall (formula, per species): 0.59
Ave. precision (formula, per species): 0.55
Ave. recall (exact, per species): 0.46
Ave. precision (exact, per species): 0.45
Ave. total time (per model): 14.76
Ave. total time (per element, per model): 0.47
Ave. LLM time (per model): 13.68
Ave. LLM time (per element, per model): 0.43
Average number of predictions per species: 1.49


In [4]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_autoType_updated.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 683
Number of models with predictions: 635
Number of annotations evaluated: 23765
Average accuracy (per model): 0.30
Ave. recall (formula): 0.26
Ave. precision (formula): 0.24
Ave. recall (exact): 0.26
Ave. precision (exact): 0.15
Average accuracy (per species): 0.44
Ave. recall (formula, per species): 0.42
Ave. precision (formula, per species): 0.40
Ave. recall (exact, per species): 0.28
Ave. precision (exact, per species): 0.28
Ave. total time (per model): 15.61
Ave. total time (per element, per model): 0.45
Ave. LLM time (per model): 14.62
Ave. LLM time (per element, per model): 0.42
Average number of predictions per species: 1.20


In [11]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_autoType.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 683
Number of models with predictions: 617
Number of annotations evaluated: 23765
Average accuracy (per model): 0.28
Ave. recall (formula): 0.25
Ave. precision (formula): 0.23
Ave. recall (exact): 0.24
Ave. precision (exact): 0.14
Average accuracy (per species): 0.44
Ave. recall (formula, per species): 0.42
Ave. precision (formula, per species): 0.40
Ave. recall (exact, per species): 0.28
Ave. precision (exact, per species): 0.28
Ave. total time (per model): 17.26
Ave. total time (per element, per model): 0.50
Ave. LLM time (per model): 16.18
Ave. LLM time (per element, per model): 0.46
Average number of predictions per species: 1.10


# Type distribution
Can LLM accurately identify complex?

In [4]:
def evaluate_complex_detection(csv_path: str):
    """
    Evaluate precision and recall for detecting complexes.
    
    Ground truth:
        qualifier == 'hasPart'  → complex
    Prediction:
        detected_entity_type == 'complex'
    """
    df = pd.read_csv(csv_path)

    # Ground truth: True if this species is actually a complex
    gt = df['qualifier'].astype(str).str.contains('hasPart', case=False, na=False)

    # Prediction: True if model predicted complex
    pred = df['detected_entity_type'].astype(str).str.lower() == 'complex'

    # Calculate TP, FP, FN
    tp = ((pred == True) & (gt == True)).sum()
    fp = ((pred == True) & (gt == False)).sum()
    fn = ((pred == False) & (gt == True)).sum()
    tn = ((pred == False) & (gt == False)).sum()

    # Precision and Recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    print(f"Total species: {len(df)}, 'hasPart' species: {sum(gt)}, 'complex' predictions: {sum(pred)}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, Accuracy: {accuracy:.4f}")

evaluate_complex_detection("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType_complex_improved.csv")

Total species: 5292, 'hasPart' species: 2068, 'complex' predictions: 2117
Precision: 0.8375, Recall: 0.8574, Accuracy: 0.8793


In [5]:
evaluate_complex_detection("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts_updated.csv")

Total species: 12486, 'hasPart' species: 1043, 'complex' predictions: 967
Precision: 0.7901, Recall: 0.7325, Accuracy: 0.9614


In [6]:
evaluate_complex_detection("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_9606_autoType.csv")

Total species: 17510, 'hasPart' species: 2843, 'complex' predictions: 2855
Precision: 0.8252, Recall: 0.8287, Accuracy: 0.9437


# Statistics

In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts_updated.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 342
Number of models with predictions: 331
Number of annotations evaluated: 12486
Average accuracy (per model): 0.76
Ave. recall (formula): 0.77
Ave. precision (formula): 0.58
Ave. recall (exact): 0.65
Ave. precision (exact): 0.23
Average accuracy (per species): 0.87
Ave. recall (formula, per species): 0.87
Ave. precision (formula, per species): 0.65
Ave. recall (exact, per species): 0.57
Ave. precision (exact, per species): 0.36
Ave. total time (per model): 18.66
Ave. total time (per element, per model): 0.51
Ave. LLM time (per model): 18.10
Ave. LLM time (per element, per model): 0.50
Average number of predictions per species: 2.76


In [6]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 312
Number of annotations evaluated: 11373
Average accuracy (per model): 0.86
Ave. recall (formula): 0.86
Ave. precision (formula): 0.65
Ave. recall (exact): 0.73
Ave. precision (exact): 0.26
Average accuracy (per species): 0.93
Ave. recall (formula, per species): 0.93
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.60
Ave. precision (exact, per species): 0.39
Ave. total time (per model): 19.55
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.97
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 2.77


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 308
Number of annotations evaluated: 11370
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.64
Ave. recall (exact): 0.72
Ave. precision (exact): 0.25
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.67
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.37
Ave. total time (per model): 19.25
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.57
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.79


In [9]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 313
Number of annotations evaluated: 11373
Average accuracy (per model): 0.89
Ave. recall (formula): 0.89
Ave. precision (formula): 0.23
Ave. recall (exact): 0.84
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.95
Ave. precision (formula, per species): 0.26
Ave. recall (exact, per species): 0.68
Ave. precision (exact, per species): 0.15
Ave. total time (per model): 19.59
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.90
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 8.93


In [8]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 306
Number of annotations evaluated: 11370
Average accuracy (per model): 0.87
Ave. recall (formula): 0.87
Ave. precision (formula): 0.23
Ave. recall (exact): 0.82
Ave. precision (exact): 0.09
Average accuracy (per species): 0.93
Ave. recall (formula, per species): 0.93
Ave. precision (formula, per species): 0.25
Ave. recall (exact, per species): 0.67
Ave. precision (exact, per species): 0.14
Ave. total time (per model): 18.79
Ave. total time (per element, per model): 0.53
Ave. LLM time (per model): 18.12
Ave. LLM time (per element, per model): 0.51
Average number of predictions per species: 8.97


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 305
Number of models with predictions: 305
Number of annotations evaluated: 10850
Average accuracy (per model): 0.93
Ave. recall (formula): 0.93
Ave. precision (formula): 0.24
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.96
Ave. recall (formula, per species): 0.96
Ave. precision (formula, per species): 0.26
Ave. recall (exact, per species): 0.69
Ave. precision (exact, per species): 0.15
Ave. total time (per model): 19.60
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.89
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 9.28


In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 307
Number of models with predictions: 307
Number of annotations evaluated: 10814
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.69
Ave. recall (exact): 0.77
Ave. precision (exact): 0.27
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.60
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.96
Ave. total time (per element, per model): 0.57
Ave. LLM time (per model): 19.26
Ave. LLM time (per element, per model): 0.55
Average number of predictions per species: 2.88


In [4]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 308
Number of annotations evaluated: 11370
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.64
Ave. recall (exact): 0.72
Ave. precision (exact): 0.25
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.67
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.37
Ave. total time (per model): 19.25
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.57
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.79


In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 308
Number of models with predictions: 308
Number of annotations evaluated: 10815
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.68
Ave. recall (exact): 0.77
Ave. precision (exact): 0.27
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.61
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.98
Ave. total time (per element, per model): 0.57
Ave. LLM time (per model): 19.28
Ave. LLM time (per element, per model): 0.55
Average number of predictions per species: 2.88


In [14]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 316
Number of models with predictions: 310
Number of annotations evaluated: 11154
Average accuracy (per model): 0.88
Ave. recall (formula): 0.88
Ave. precision (formula): 0.66
Ave. recall (exact): 0.75
Ave. precision (exact): 0.26
Average accuracy (per species): 0.94
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.61
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.10
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.40
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.83


In [ ]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 315
Number of models with predictions: 309
Number of annotations evaluated: 11145
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.24
Ave. recall (exact): 0.85
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.95
Ave. precision (formula, per species): 0.25
Ave. recall (exact, per species): 0.69
Ave. precision (exact, per species): 0.14
Ave. total time (per model): 19.39
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.64
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 9.20


In [4]:
print_evaluation_results("autoType/biomd251106_chebi_direct_llama-3_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 306
Number of models with predictions: 301
Number of annotations evaluated: 10806
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.73
Ave. recall (exact): 0.79
Ave. precision (exact): 0.36
Average accuracy (per species): 0.90
Ave. recall (formula, per species): 0.90
Ave. precision (formula, per species): 0.77
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.48
Ave. total time (per model): 25.95
Ave. total time (per element, per model): 0.73
Ave. LLM time (per model): 23.66
Ave. LLM time (per element, per model): 0.67
Average number of predictions per species: 2.77


In [6]:
print_evaluation_results("results/biomd251106_chebi_direct_llama-3_top10.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 316
Number of annotations evaluated: 11370
Average accuracy (per model): 0.87
Ave. recall (formula): 0.87
Ave. precision (formula): 0.76
Ave. recall (exact): 0.79
Ave. precision (exact): 0.36
Average accuracy (per species): 0.87
Ave. recall (formula, per species): 0.86
Ave. precision (formula, per species): 0.74
Ave. recall (exact, per species): 0.57
Ave. precision (exact, per species): 0.46
Ave. total time (per model): 25.23
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 22.73
Ave. LLM time (per element, per model): 0.64
Average number of predictions per species: 2.67


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-3_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 307
Number of models with predictions: 307
Number of annotations evaluated: 10826
Average accuracy (per model): 0.94
Ave. recall (formula): 0.94
Ave. precision (formula): 0.23
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.68
Ave. precision (exact, per species): 0.13
Ave. total time (per model): 24.71
Ave. total time (per element, per model): 0.70
Ave. LLM time (per model): 23.80
Ave. LLM time (per element, per model): 0.67
Average number of predictions per species: 9.94


In [8]:
print_evaluation_results("results/biomd251106_chebi_rag_llama-3_top10.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 320
Number of annotations evaluated: 11370
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.23
Ave. recall (exact): 0.85
Ave. precision (exact): 0.09
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.65
Ave. precision (exact, per species): 0.12
Ave. total time (per model): 23.79
Ave. total time (per element, per model): 0.67
Ave. LLM time (per model): 22.84
Ave. LLM time (per element, per model): 0.64
Average number of predictions per species: 9.89


In [8]:
# find different model, species pairs
df1 = pd.read_csv('results/biomd251106_chebi_rag_llama-4_top3.csv')
df2 = pd.read_csv('autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated.csv')
# only consider is, isVersionOf
df1 = df1[df1['qualifier'].isin(['is', 'isVersionOf'])]
df2 = df2[df2['qualifier'].isin(['is', 'isVersionOf'])]

# find pairs in df1 but not in df2
df1['model_species_pair'] = df1['model'] + '_' + df1['species_id']
df2['model_species_pair'] = df2['model'] + '_' + df2['species_id']
df1[~df1['model_species_pair'].isin(df2['model_species_pair'])].to_csv('autoType/biomd251106_chebi_rag_llama-4_top3_auto_diff.csv', index=False)